# Orchestrate: 이슈에서 병합된 PR까지

이 노트북은 현실적인 종단 간 루프를 따라 에이전트를 이끌어 갑니다. 모호한 버그 리포트를 읽고, 버그를 찾고, 고치고, PR을 열고, CI를 통과하고, 리뷰 피드백에 대응하고, 병합합니다. 실제 메인테이너의 작업은 결코 선형적이지 않은데, 바로 그것이 이 연습의 요점입니다. 에이전트는 여러 종류의 도구(JSON 읽기, 코드 grep, 파일 편집, 모의 CLI 실행, CI 출력 파싱)에 걸쳐 상태를 이어 가면서, 도중에 나타나는 두 가지 돌발 상황(CI 실패, 그리고 독스트링을 요구하는 리뷰 봇)에서 회복해야 합니다.

상태는 이슈 본문 → 파일 경로 → 수정 diff → PR 번호 → CI 출력 → 리뷰 코멘트 → 최종 병합 순으로 체인을 따라 흐릅니다. 픽스처의 `gh-mock` CLI가 모든 것을 `.gh-state/`에 저장하므로, 각 단계가 앞선 단계의 결과를 볼 수 있고 네트워크 없이도 실제 GitHub 워크플로를 흉내 낼 수 있습니다.

iterate 노트북에 더해 이 노트북이 가르쳐 주는 것:

- **긴 체인에 걸친 멀티턴 지시.** 세션 파일 시스템과 대화 기록이 턴을 넘어 유지되므로, 사용자 메시지마다 직전에 멈춘 지점에서 이어집니다. 마지막에 최종 상태를 검증할 때 이를 활용합니다.
- **체인 도중의 회복.** 에이전트는 CI 실패나 리뷰 코멘트를 읽고 적응해야 하며, 무작정 재시도해서는 안 됩니다.

픽스처는 `example_data/orchestrate/`에 있으며, 모의 `gh` CLI, 이슈 JSON 파일, 그리고 버그가 심어진 `src/` + `tests/` 구조가 들어 있습니다.

In [ ]:
import io
import os
import zipfile
from pathlib import Path

from anthropic import Anthropic
from utilities import stream_until_end_turn, wait_for_idle_status

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")
GH_TOKEN = os.environ.get("GITHUB_TOKEN")  # only needed for the github_repository sidebar

client = Anthropic()
FIXTURE = Path("example_data") / "orchestrate"

## 1. 픽스처 묶기

모의 저장소에는 몇 개의 파일이 들어 있습니다. `src/url_utils.py`에 실제 버그가 있고, `src/blog.py`는 버그를 관찰하기 쉽게 해 주는 호출부이며, `tests/test_urls.py`는 버그가 고쳐질 때까지 실패합니다. `gh-mock` CLI와 `issue_42.json`이 에이전트가 구동할 GitHub 유사 워크플로를 제공합니다. 디렉터리를 메모리에서 zip으로 묶어 단일 파일 리소스로 업로드합니다. 실제 GitHub 저장소를 마운트하는 방법은 마지막 사이드바를 참고하세요.

In [ ]:
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as zf:
    for f in FIXTURE.rglob("*"):
        if f.is_file() and f.name != "README.md":
            zf.write(f, f.relative_to(FIXTURE))
buf.seek(0)
fixture_zip = client.beta.files.upload(file=("repo.zip", buf, "application/zip"))
print(f"fixture: {fixture_zip.id}")

## 2. 에이전트 + 환경 + 세션

환경에 `pytest`를 pip 의존성으로 선언해 두어, 에이전트가 CI 루프의 일부로 실제 테스트 스위트를 실행할 수 있게 합니다. 이 쿡북에서 패키지 설치를 위해 네트워크 접근이 필요한 첫 노트북이며, 그래서 나머지는 `limited` 네트워킹 설정을 쓰면서 `allow_package_managers: True`를 함께 지정합니다.

In [ ]:
agent = client.beta.agents.create(
    name="cookbook-orchestrate",
    model=MODEL,
    system=(
        "You are a maintainer bot. You read issues via `./gh-mock`, explore "
        "the codebase, write fixes, and shepherd PRs through CI and review. "
        "When CI fails or a reviewer requests changes, read what they said "
        "and address it, don't just retry blindly.\n\n"
        "Work in /mnt/user. All gh-mock commands run from there."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        }
    ],
)

env = client.beta.environments.create(
    name="cookbook-orchestrate-env",
    config={
        "type": "cloud",
        "networking": {"type": "limited", "allow_package_managers": True},
        "packages": {"pip": ["pytest"]},
    },
)

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[{"type": "file", "file_id": fixture_zip.id, "mount_path": "repo.zip"}],
    title="Issue #42 → PR",
)
print(f"session: {session.id}")

## 3. 전체 체인 실행하기

지시 한 번으로 전체 루프가 시작됩니다. 눈여겨볼 회복 지점은 두 곳, CI 실패와 리뷰 코멘트입니다. 에이전트의 첫 수정이 불완전하면(예: `é`만 처리하고 `ü`를 놓치면) `gh-mock pr checks`가 0이 아닌 코드로 종료되며 pytest 출력을 남기고, 에이전트는 그것을 읽고 개선해야 합니다. CI가 초록불이 된 뒤에는 `slugify()`에 독스트링이 없으면 리뷰어 봇이 병합을 막아, 최종 병합 전에 에이전트에 적응할 기회를 한 번 더 줍니다.

In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Unpack /mnt/session/uploads/repo.zip into /mnt/user "
                        "and ship a fix for issue #42 end-to-end. Read the "
                        "./gh-mock script first to see what subcommands it "
                        "supports; use those to view the issue, open a PR, "
                        "run CI, handle review feedback, and merge. Show me "
                        "the final PR state when you're done."
                    ),
                }
            ],
        }
    ],
)

print("=== full orchestrate chain ===")
stream_until_end_turn(client, session.id)

## 4. 멀티턴 검증

세션은 상태를 유지합니다. 컨테이너 파일 시스템과 대화 기록이 턴을 넘어 유지되므로, 후속 작업은 `user.message`를 하나 더 보내기만 하면 됩니다. 여기서는 이를 활용해 최종 상태를 독립적으로 검증합니다. 모의 CLI가 PR 상태를 `.gh-state/pr_101.json`에 저장하므로, 이 파일을 출력하는 것이 CI가 초록불이고 병합 전에 최소 한 건의 리뷰 승인이 있었으며 PR이 병합되었는지 확인하는 가장 간단한 방법입니다.

In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Print the contents of /mnt/user/.gh-state/pr_101.json so "
                        "I can see the final state, CI status, and reviews."
                    ),
                }
            ],
        }
    ],
)
stream_until_end_turn(client, session.id)

wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
print("archived")

## 사이드바: 실제 GitHub 저장소 마운트하기

위 픽스처는 노트북을 오프라인에서 실행할 수 있고 GitHub 자격 증명 없이도 시험해 볼 수 있도록 만든 모의 환경입니다. 실제 저장소로 실제 작업을 하려면 위의 `{"type": "file", ...}` 마운트를 `{"type": "github_repository", ...}` 리소스로 바꾸면, API가 세션 시작 시 저장소를 컨테이너로 클론합니다.

iterate 노트북의 파일 마운트와 같은 `resources=` 필드이며, 이 리스트는 여러 타입을 섞어 받을 수 있으므로 한 번의 호출로 저장소를 클론하면서 별도 설정 파일도 함께 마운트할 수 있습니다. 에이전트의 bash/read/grep 도구는 작업 트리를 평범한 디렉터리로 봅니다. 다른 점은 클론을 여러분 대신 API가 처리한다는 것뿐입니다.

```python
session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[
        {
            "type": "github_repository",
            "url": "https://github.com/anthropics/claude-cookbooks",
            "mount_path": "/workspace/cookbook",
            "authorization_token": GH_TOKEN,
            "checkout": {"type": "branch", "name": "main"},
        }
    ],
    title="Repo explorer",
)
```

비공개 저장소는 물론이고 클론을 인증하기 위해서도 `GITHUB_TOKEN`이 필요합니다. 클론은 세션 생성 시 한 번만 일어나며, 이후 턴은 다시 클론하지 않고 같은 작업 트리에서 동작합니다.